# Visualization & Evaluation — SlotODEModel (JAX / Equinox)

Load a trained checkpoint and inspect reconstructions, slot masks, ODE trajectories,
attention dynamics, and compute ARI-FG / mIoU metrics using CLEVR-with-masks ground truth.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import jax
import jax.numpy as jnp
import equinox as eqx
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment

from model import SlotODEModel
from train import build_tfrecord_dataset, tf_to_jax

RESOLUTION = (64, 64)
CKPT_PATH = 'checkpoints/best.eqx'
TFRECORDS_PATH = 'clevr_with_masks_clevr_with_masks_train.tfrecords'

print(f'JAX devices: {jax.devices()}')

%matplotlib inline

JAX devices: [CudaDevice(id=0)]


In [5]:
# --- load checkpoint ---
import pickle

meta_path = CKPT_PATH.replace('.eqx', '_meta.pkl')
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)

args = meta['args']
print(f'Checkpoint from step {meta["step"]}, best_val_loss={meta["best_val_loss"]:.5f}')
print(f'Args: {args}')

key = jax.random.key(0)
model = SlotODEModel(
    resolution=RESOLUTION,
    num_slots=args['num_slots'],
    slot_dim=args['slot_dim'],
    enc_hidden_dim=args['enc_hidden_dim'],
    num_iter=args['num_iter'],
    solver=args.get('solver', 'tsit5'),
    key=key,
)
model = eqx.tree_deserialise_leaves(CKPT_PATH, model)

n_params = sum(p.size for p in jax.tree.leaves(eqx.filter(model, eqx.is_array)))
print(f'Model loaded ({n_params:,} params)')

Checkpoint from step 5000, best_val_loss=0.00475
Args: {'model': 'slot_ode', 'data_root': '../CLEVR_v1.0', 'batch_size': 64, 'lr': 0.0004, 'warmup_steps': 2000, 'total_steps': 50000, 'num_slots': 7, 'slot_dim': 64, 'enc_hidden_dim': 64, 'num_iter': 3, 'log_every': 100, 'val_every': 5000, 'img_every': 5000, 'ckpt_every': 10000, 'ckpt_dir': 'checkpoints', 'experiment': 'slot_ode_jax', 'run_name': None, 'num_workers': 4, 'seed': 42, 'grad_clip': 1.0, 'resume': None}
Model loaded (775,108 params)


In [6]:
# --- load sample images + GT masks from TFRecords (val split) ---

# count total to find val offset
print("Counting records...")
total = sum(1 for _ in tf.data.TFRecordDataset(TFRECORDS_PATH, compression_type='GZIP'))
val_offset = total - 5000
print(f"Total: {total}, val starts at: {val_offset}")

# load a small batch from val split
val_ds = build_tfrecord_dataset(
    TFRECORDS_PATH, RESOLUTION[0], batch_size=4,
    shuffle=False, skip=val_offset, take=100,
)

for batch in val_ds.take(1):
    imgs_tf, masks_tf, vis_tf = batch

imgs = tf_to_jax(imgs_tf)          # [4, 3, 64, 64]
gt_masks_raw = masks_tf.numpy()     # [4, 11, 64, 64] uint8 binary
gt_vis = vis_tf.numpy()             # [4, 11]

# convert binary per-object masks to segmentation map
def masks_to_seg(masks, visibility):
    """[11, H, W] binary uint8 + [11] vis -> [H, W] int seg map."""
    H, W = masks.shape[1], masks.shape[2]
    seg = np.zeros((H, W), dtype=np.int32)
    for i in range(1, 11):
        if visibility[i] > 0.5:
            seg[masks[i] > 127] = i
    return seg

gt_segs = [masks_to_seg(gt_masks_raw[i], gt_vis[i]) for i in range(4)]

print(f'Input batch shape: {imgs.shape}')
print(f'GT mask unique values (first image): {np.unique(gt_segs[0])}')

Counting records...
Total: 100000, val starts at: 95000


2026-03-06 06:28:50.130391: E tensorflow/core/util/util.cc:131] oneDNN supports DT_UINT8 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Input batch shape: (4, 3, 64, 64)
GT mask unique values (first image): [0 1 2 3 4]


In [7]:
# --- forward pass with trajectory ---
key = jax.random.key(42)
recon, masks, slots, traj = model(imgs, key=key, return_traj=True)

print(f'recon:   {recon.shape}')    # [4, 3, 64, 64]
print(f'masks:   {masks.shape}')    # [4, 7, 64, 64]
print(f'slots:   {slots.shape}')    # [4, 7, 64]
print(f'traj:    {traj.shape}')     # [20, 4, 7, 64]

# convert to numpy for plotting
imgs_np = np.array(imgs)
recon_np = np.array(recon)
masks_np = np.array(masks)
slots_np = np.array(slots)
traj_np = np.array(traj)

B = imgs_np.shape[0]
N_slots = masks_np.shape[1]
T_steps = traj_np.shape[0]
T_ode = np.linspace(0, model.slot_attention_ode.T, T_steps)

E0306 06:32:33.906100    3849 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0306 06:32:34.068210    3849 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


recon:   (4, 3, 64, 64)
masks:   (4, 7, 64, 64)
slots:   (4, 7, 64)
traj:    (20, 4, 7, 64)


## 1. Reconstruction vs Input

In [8]:
fig, axes = plt.subplots(2, B, figsize=(4 * B, 8))

for j in range(B):
    inp = imgs_np[j].transpose(1, 2, 0)     # CHW -> HWC
    rec = recon_np[j].transpose(1, 2, 0).clip(0, 1)
    axes[0, j].imshow(inp)
    axes[0, j].set_title('Input')
    axes[0, j].axis('off')
    axes[1, j].imshow(rec)
    axes[1, j].set_title(f'Recon (MSE={((inp - rec)**2).mean():.5f})')
    axes[1, j].axis('off')

plt.suptitle('Input vs Reconstruction', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/745530134.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Per-Slot Masks

In [9]:
fig, axes = plt.subplots(B, N_slots + 1, figsize=(2.5 * (N_slots + 1), 2.5 * B))
if B == 1:
    axes = axes[None, :]

for i in range(B):
    axes[i, 0].imshow(imgs_np[i].transpose(1, 2, 0))
    axes[i, 0].set_title('Input')
    axes[i, 0].axis('off')
    for s in range(N_slots):
        axes[i, s + 1].imshow(masks_np[i, s], cmap='viridis', vmin=0, vmax=1)
        axes[i, s + 1].set_title(f'Slot {s}')
        axes[i, s + 1].axis('off')

plt.suptitle('Per-Slot Attention Masks', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/2172141240.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Per-Slot Reconstructions (masked)

In [10]:
# decode each slot individually using the decoder
def decode_single_slot(model, slot):
    """Decode a single slot [D] -> [3, H, W] RGB."""
    return model.dec.decode_single(slot)[:3]  # drop mask channel

slot_recons_all = []
for i in range(B):
    # vmap over slots for this image
    decode_slots = jax.vmap(lambda s: decode_single_slot(model, s))
    decoded = decode_slots(slots[i])  # [N_slots, 3, H, W]
    slot_recons_all.append(np.array(decoded))

fig, axes = plt.subplots(B, N_slots + 2, figsize=(2.5 * (N_slots + 2), 2.5 * B))
if B == 1:
    axes = axes[None, :]

for i in range(B):
    axes[i, 0].imshow(imgs_np[i].transpose(1, 2, 0))
    axes[i, 0].set_title('Input')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(recon_np[i].transpose(1, 2, 0).clip(0, 1))
    axes[i, 1].set_title('Recon')
    axes[i, 1].axis('off')
    for s in range(N_slots):
        sr = slot_recons_all[i][s].transpose(1, 2, 0).clip(0, 1)
        m = masks_np[i, s][:, :, None]
        axes[i, s + 2].imshow(sr * m)
        axes[i, s + 2].set_title(f'Slot {s}')
        axes[i, s + 2].axis('off')

plt.suptitle('Per-Slot Masked Reconstructions', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/1531354113.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Slot Trajectories (ODE dynamics)

In [11]:
# plot L2 norm of each slot over ODE integration time
fig, axes = plt.subplots(1, B, figsize=(5 * B, 3))
if B == 1:
    axes = [axes]

for i in range(B):
    norms = np.linalg.norm(traj_np[:, i, :, :], axis=-1)  # [T, N_slots]
    for s in range(N_slots):
        axes[i].plot(T_ode, norms[:, s], label=f'Slot {s}')
    axes[i].set_xlabel('ODE time t')
    axes[i].set_ylabel('||slot||_2')
    axes[i].set_title(f'Image {i}')
    axes[i].legend(fontsize=6, ncol=2)

plt.suptitle('Slot L2 Norm over ODE Trajectory', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/4115255529.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Attention Analysis

To analyze attention over ODE time, we re-run the ODE and manually log attention at each solver step.
We do this by calling the ODE func directly at each trajectory point.

In [12]:
# compute attention at each trajectory time point
# first, get the precomputed K, V from the encoder features
enc_feat = model.encode(imgs)  # [B, H*W, D]

sa = model.slot_attention_ode
feat_norm = jax.vmap(jax.vmap(sa.norm_input))(enc_feat)
feat = jax.vmap(jax.vmap(sa.fc_input))(feat_norm)
k_proj = jax.vmap(jax.vmap(sa.to_k))(feat)
v_proj = jax.vmap(jax.vmap(sa.to_v))(feat)

# compute attention at each trajectory point
ode_func = sa.slot_ode_func
scale = ode_func.scale

att_log = []
for t_idx in range(T_steps):
    s = traj[t_idx]  # [B, N_slots, D]
    s_norm = jax.vmap(jax.vmap(ode_func.norm_slots))(s)
    q = jax.vmap(jax.vmap(ode_func.to_q))(s_norm)
    att_logits = jnp.einsum('bnd,bmd->bnm', q, k_proj) * scale
    att = jax.nn.softmax(att_logits, axis=1)
    att = att / (att.sum(axis=-1, keepdims=True) + 1e-8)
    att_log.append((T_ode[t_idx], np.array(att)))

print(f'Computed attention at {len(att_log)} time points')

Computed attention at 20 time points


### 5a. Attention Entropy over ODE Time
Low entropy = sharp attention (each feature strongly assigned to one slot).  
Should decrease over integration time as slots specialize.

In [13]:
times = [entry[0] for entry in att_log]
entropies = []  # [T, B, N_slots]

for t_val, att in att_log:
    att_clamped = np.clip(att, 1e-10, None)
    H = -(att_clamped * np.log(att_clamped)).sum(axis=-1)  # [B, N_slots]
    entropies.append(H)

entropies = np.array(entropies)  # [T, B, N_slots]

fig, axes = plt.subplots(1, B, figsize=(5 * B, 3))
if B == 1:
    axes = [axes]

for i in range(B):
    for s in range(N_slots):
        axes[i].plot(times, entropies[:, i, s], label=f'Slot {s}', alpha=0.7)
    axes[i].set_xlabel('ODE time t')
    axes[i].set_ylabel('Entropy')
    axes[i].set_title(f'Image {i}')
    axes[i].legend(fontsize=6, ncol=2)

plt.suptitle('Per-Slot Attention Entropy over ODE Time', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/826890292.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5b. Attention Sharpening (max attention per feature)
How confidently each spatial feature is assigned to its "winning" slot over time.

In [14]:
max_atts = []
for t_val, att in att_log:
    max_per_feat = att.max(axis=1)  # [B, N_feat]
    max_atts.append(max_per_feat.mean(axis=-1))  # [B]

max_atts = np.array(max_atts)  # [T, B]

plt.figure(figsize=(6, 3))
for i in range(B):
    plt.plot(times, max_atts[:, i], label=f'Image {i}')
plt.xlabel('ODE time t')
plt.ylabel('Mean max attention')
plt.title('Attention Sharpening over ODE Time')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/3350592777.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Slot Competition Heatmap
Visualize how attention weights (reshaped to spatial grid) evolve at early vs late ODE time.

In [15]:
img_idx = 0
H, W = RESOLUTION
n_timepoints = 4
indices = np.linspace(0, len(att_log) - 1, n_timepoints, dtype=int)

fig, axes = plt.subplots(n_timepoints, N_slots, figsize=(2.5 * N_slots, 2.5 * n_timepoints))

for row, idx in enumerate(indices):
    t_val, att = att_log[idx]
    att_spatial = att[img_idx].reshape(N_slots, H, W)  # [N_slots, H, W]
    for s in range(N_slots):
        axes[row, s].imshow(att_spatial[s], cmap='hot', vmin=0)
        axes[row, s].axis('off')
        if row == 0:
            axes[row, s].set_title(f'Slot {s}', fontsize=9)
    axes[row, 0].set_ylabel(f't={t_val:.2f}', fontsize=9)

plt.suptitle(f'Slot Attention Maps over ODE Time (Image {img_idx})', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/1857755868.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Phase Portrait — Slot Dynamics in PCA Space
Project the 64-dim slot trajectories onto their top 2 principal components. If slots converge to fixed points (attractors), trajectories will spiral/flow inward.

In [16]:
from sklearn.decomposition import PCA

# fit PCA on all slot states across time, images, and slots
all_states = traj_np.reshape(-1, traj_np.shape[-1])  # [T*B*N_slots, 64]
pca = PCA(n_components=2)
pca.fit(all_states)
print(f'PCA explained variance: {pca.explained_variance_ratio_[0]:.3f}, {pca.explained_variance_ratio_[1]:.3f}')

traj_2d = pca.transform(all_states).reshape(T_steps, B, N_slots, 2)

colors = plt.cm.tab10(np.linspace(0, 1, N_slots))

fig, axes = plt.subplots(1, B, figsize=(6 * B, 6))
if B == 1:
    axes = [axes]

for i in range(B):
    ax = axes[i]
    for s in range(N_slots):
        xs = traj_2d[:, i, s, 0]
        ys = traj_2d[:, i, s, 1]

        ax.plot(xs, ys, color=colors[s], alpha=0.4, linewidth=1)

        # arrows showing flow direction
        n_arrows = 5
        arrow_idx = np.linspace(0, T_steps - 2, n_arrows, dtype=int)
        for ai in arrow_idx:
            dx = xs[ai + 1] - xs[ai]
            dy = ys[ai + 1] - ys[ai]
            ax.annotate('', xy=(xs[ai] + dx, ys[ai] + dy), xytext=(xs[ai], ys[ai]),
                        arrowprops=dict(arrowstyle='->', color=colors[s], lw=1.5))

        ax.scatter(xs[0], ys[0], color=colors[s], marker='o', s=60, zorder=5,
                   edgecolors='black', linewidths=0.5)
        ax.scatter(xs[-1], ys[-1], color=colors[s], marker='*', s=120, zorder=5,
                   edgecolors='black', linewidths=0.5, label=f'Slot {s}')

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'Image {i}')
    ax.legend(fontsize=7, ncol=2, loc='best')
    ax.set_aspect('equal', adjustable='datalim')

plt.suptitle('Phase Portrait — Slot Trajectories in PCA Space\n(o = t=0, * = t=T)', fontsize=13)
plt.tight_layout()
plt.show()

PCA explained variance: 0.246, 0.134


/tmp/ipykernel_3714/871374308.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Velocity Magnitude — Convergence to Attractors
If slots reach a fixed point, velocity ||ds/dt|| should decay toward zero.

In [17]:
dt = T_ode[1] - T_ode[0]
velocities = np.diff(traj_np, axis=0) / dt  # [T-1, B, N_slots, 64]
vel_mag = np.linalg.norm(velocities, axis=-1)  # [T-1, B, N_slots]
t_mid = 0.5 * (T_ode[:-1] + T_ode[1:])

fig, axes = plt.subplots(1, B, figsize=(5 * B, 3))
if B == 1:
    axes = [axes]

for i in range(B):
    for s in range(N_slots):
        axes[i].plot(t_mid, vel_mag[:, i, s], color=colors[s], label=f'Slot {s}')
    axes[i].set_xlabel('ODE time t')
    axes[i].set_ylabel('||ds/dt||')
    axes[i].set_title(f'Image {i}')
    axes[i].legend(fontsize=6, ncol=2)

plt.suptitle('Slot Velocity Magnitude — Decay → Attractor', fontsize=13)
plt.tight_layout()
plt.show()

/tmp/ipykernel_3714/3543456449.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
